In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [4]:
# path file di HDFS (sesuaikan dengan folder tempat yang diunggah sebelumnya)
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"

# membaca data langsung dari HDFS menggunakan PySpark
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(hdfs_path)

# 1. menampilkan printSchema()
print("---SKEMA DATA---")
df_raw.printSchema()

# 2. menampilkan jumlah baris (count())
jumlah_baris = df_raw.count()
print(f"\nJumlah baris total dalam dataset: {jumlah_baris}")

# 3. menampilkan 10 baris pertama (show(10))
print("\n--- 10 BARIS PERTAMA ---")
df_raw.show(10, truncate=False)

---SKEMA DATA---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)


Jumlah baris total dalam dataset: 1000

--- 10 BARIS PERTAMA ---
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0 

In [5]:
from pyspark.sql.functions import col, when, lit, sum as _sum, avg, count

print("---CEK JUMLAH DATA KOSONG PADA KOLOM RATING---")
df_raw.select([_sum(when(col('rating').isNull(), 1).otherwise(0)).alias('missing_rating')]).show()

# menangani data kosong dengan mengisi nilai default (misal: 3.0)
df_clean = df_raw.na.fill({"rating": 3.0})

print("Data kosong berhasil ditangani!")

---CEK JUMLAH DATA KOSONG PADA KOLOM RATING---
+--------------+
|missing_rating|
+--------------+
|           204|
+--------------+

Data kosong berhasil ditangani!


In [6]:
from pyspark.sql.functions import col, when

# 1. menambahkan kolom total_pendapatan (unit_terjual * harga_satuan)
# 2. menambahkan kolom tier_transaksi ("Besar" jika > 500000, selain itu "Kecil")
df_transformed = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn(
        "tier_transaksi",
        when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
    )

# menampilkan 5 baris hasil transformasi
df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(5, truncate=False)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|3           |90000       |270000          |Kecil         |
|ORD-3001|3           |200000      |600000          |Besar         |
|ORD-3002|8           |60000       |480000          |Kecil         |
|ORD-3003|6           |350000      |2100000         |Besar         |
|ORD-3004|10          |60000       |600000          |Besar         |
+--------+------------+------------+----------------+--------------+
only showing top 5 rows



In [7]:
from pyspark.sql.functions import desc, round

# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
print("1. Kategori dengan Total Pendapatan Tertinggi:")
df_transformed.groupBy("kategori") \
    .sum("total_pendapatan") \
    .withColumnRenamed("sum(total_pendapatan)", "total_pendapatan_kategori") \
    .orderBy(desc("total_pendapatan_kategori")) \
    .show(1, truncate=False)

# 2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
print("2. Kota dengan Jumlah Transaksi Tier 'Besar' Terbanyak:")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .withColumnRenamed("count", "jumlah_transaksi_besar") \
    .orderBy(desc("jumlah_transaksi_besar")) \
    .show(1, truncate=False)

# 3. Berapa rata-rata rating untuk masing-masing metode_pembayaran?
print("3. Rata-rata rating berdasarkan Metode Pembayaran:")
df_transformed.groupBy("metode_pembayaran") \
    .agg(round(avg("rating"), 2).alias("rata_rata_rating")) \
    .orderBy(desc("rata_rata_rating")) \
    .show(truncate=False)

1. Kategori dengan Total Pendapatan Tertinggi:
+------------+-------------------------+
|kategori    |total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|138665000                |
+------------+-------------------------+
only showing top 1 row

2. Kota dengan Jumlah Transaksi Tier 'Besar' Terbanyak:
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|92                    |
+----+----------------------+
only showing top 1 row

3. Rata-rata rating berdasarkan Metode Pembayaran:
+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|COD              |3.95            |
|Transfer Bank    |3.93            |
|E-Wallet         |3.9             |
|Kartu Kredit     |3.86            |
+-----------------+----------------+



In [8]:
# menentukan path direktori penyimpanan di HDFS
output_hdfs_path = "hdfs://localhost:9000/user/uwah/tugas4/hasil_transaksi_clean"

# menyimpan DataFrame ke HDFS dalam format CSV (lengkap dengan header)
df_transformed.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_hdfs_path)

print(f"Data berhasil disimpan ke HDFS pada direktori: {output_hdfs_path}")

Data berhasil disimpan ke HDFS pada direktori: hdfs://localhost:9000/user/uwah/tugas4/hasil_transaksi_clean


In [9]:
!hdfs dfs -ls /user/uwah/tugas4/hasil_transaksi_clean

Found 2 items
-rw-r--r--   3 uwah supergroup          0 2026-09-16 14:11 /user/uwah/tugas4/hasil_transaksi_clean/_SUCCESS
-rw-r--r--   3 uwah supergroup      92296 2026-09-16 14:11 /user/uwah/tugas4/hasil_transaksi_clean/part-00000-e15fe2a6-d52e-4f00-a36e-29bfdae30583-c000.csv
